In [32]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("ddia.pdf")
documents = loader.load()
filtered_docs = [
    doc for doc in documents
    if doc.metadata.get("page", 0) >= 22
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(filtered_docs)

chunks[10].page_content







fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/MJYOHX+MyriadPro-Cond', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 66, '/FontDescriptor': IndirectObject(32, 0, 1656062365328), '/LastChar': 116, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [416, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 405, 383, 0, 0, 0, 289, 255]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


'cases), and they should all be able to work on it productively. See “Maintainabil‐\nity” on page 16.\nThese words are often cast around without a clear understanding of what they mean.\nIn the interest of thoughtful engineering, we will spend the rest of this chapter\nexploring ways of thinking about reliability, scalability and maintainability. Then, in\nthe following chapters, we will look at various techniques, architectures and algo‐\nrithms that are used in order to achieve those goals.\nReliability\nEverybody has an intuitive idea of what it means for software to be reliable or unreli‐\nable. For software, typical expectations include:\n• The application performs the function that the user expected.\n• It can tolerate the user making mistakes, or using the software in unexpected\nways.\n• Its performance is good enough for the required use case, under expected load\nand data volume.\n• The system prevents any unauthorized access and abuse.'

In [33]:
import re

def clean_pdf_text(text: str) -> str:
    # Join words split across lines by PDF line wrapping
    text = re.sub(r"(\w)[\u00ad\u2010\u2011\u2012\u2013\u2014-]\s*\n\s*(\w)", r"\1\2", text)

    # Replace remaining line breaks and repeated whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [34]:
for chunk in chunks:
  chunk.page_content=clean_pdf_text(chunk.page_content)
chunks[0].page_content

'CHAPTER 1 Reliable, Scalable and Maintainable Applications The Internet was done so well that most people think of it as a natural resource like the Pacific Ocean, rather than something that was man-made. When was the last time a technology with a scale like that was so error-free? —Alan Kay, Dr Dobb’s Journal (2012) Many applications today are data-intensive, as opposed to compute-intensive. Raw CPU power is rarely a limiting factor for these applications — bigger problems are usually the amount of data, the complexity of data, and the speed at which it is changing. A data-intensive application is typically built from standard building blocks which provide commonly needed functionality. For example, many applications need to: • Store data so that they, or another application, can find it again later (databases), • Remember the result of an expensive operation, to speed up reads (caches), • Allow users to search data by keyword or filter it in various ways ( search indexes),'

In [8]:
import ollama

result = ollama.embed(
    model="mxbai-embed-large",
    input=chunks[0].page_content
)

vector = result["embeddings"][0]

print("Vector dimensions:", len(vector))

Vector dimensions: 1024


In [36]:
import ollama

records = []

for i, chunk in enumerate(chunks):
    result = ollama.embed(
        model="mxbai-embed-large",
        input=chunk.page_content
    )

    records.append({
        "id": f"ddia-chunk-{i}",
        "values": result["embeddings"][0],
        "metadata": {
            "text": chunk.page_content,
            "source": "ddia.pdf",
            "page": chunk.metadata.get("page", 0) + 1
        }
    })
    if (len(chunks)-i) % 10 ==0:
        print(f"left:{len(chunks)-i}")



left:1520
left:1510
left:1500
left:1490
left:1480
left:1470
left:1460
left:1450
left:1440
left:1430
left:1420
left:1410
left:1400
left:1390
left:1380
left:1370
left:1360
left:1350
left:1340
left:1330
left:1320
left:1310
left:1300
left:1290
left:1280
left:1270
left:1260
left:1250
left:1240
left:1230
left:1220
left:1210
left:1200
left:1190
left:1180
left:1170
left:1160
left:1150
left:1140
left:1130
left:1120
left:1110
left:1100
left:1090
left:1080
left:1070
left:1060
left:1050
left:1040
left:1030
left:1020
left:1010
left:1000
left:990
left:980
left:970
left:960
left:950
left:940
left:930
left:920
left:910
left:900
left:890
left:880
left:870
left:860
left:850
left:840
left:830
left:820
left:810
left:800
left:790
left:780
left:770
left:760
left:750
left:740
left:730
left:720
left:710
left:700
left:690
left:680
left:670
left:660
left:650
left:640
left:630
left:620
left:610
left:600
left:590
left:580
left:570
left:560
left:550
left:540
left:530
left:520
left:510
left:500
left:490
left:480
le

In [37]:
from pinecone import Pinecone, ServerlessSpec
import os

pc = Pinecone(api_key=os.getenv("PINECONE_APIKEY"))

pc.create_index(
    name="ddia-ollama-vectors",
    dimension=1024,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

Name:,ddia-ollama-vectors
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://ddia-ollama-vectors-4lw09wz.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [38]:
index = pc.Index("ddia-ollama-vectors")

response = index.upsert(
    vectors=records,
    namespace="ddia",
    batch_size=100
)

print(response)

d:\agentic_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Upserting: 100%|██████████| 16/16 [00:18<00:00,  1.16s/batch]

UpsertResponse(SUCCESS: 1528/1528 items, 16/16 batches)


In [29]:
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1024, total_vector_count=1528, metric='cosine', namespaces=1)


In [19]:
index_info = pc.indexes.describe(name=index_name)

print(index_info.schema)

IndexSchema(fields={'title': StringField(description=None, filterable=False, full_text_search=FullTextSearchConfig(language='en', stemming=False, stop_words=False, ngram=None), embed=None, sparse_embed=None, store_text=True), 'chunk_text': StringField(description=None, filterable=False, full_text_search=FullTextSearchConfig(language='en', stemming=False, stop_words=False, ngram=None), embed=FieldEmbedConfig(model='llama-text-embed-v2', dimension=1024, metric='cosine', write_parameters={'input_type': 'passage', 'truncate': 'END'}, read_parameters={'input_type': 'query', 'truncate': 'END'}), sparse_embed=None, store_text=True)})


In [30]:
import ollama

question = "How does a database use a write-ahead log?"

result = ollama.embed(
    model="mxbai-embed-large",
    input=question
)

query_vector = result["embeddings"][0]

print("Query vector dimensions:", len(query_vector))

Query vector dimensions: 1024


In [31]:
results = index.query(
    namespace="ddia",
    vector=query_vector,
    top_k=5,
    include_metadata=True
)

for i, match in enumerate(results["matches"], start=1):
    print(f"\n--- Result {i} ---")
    print("Score:", match["score"])
    print("Page:", match["metadata"].get("page"))
    print("Text:", match["metadata"].get("text", "")[:1000])


--- Result 1 ---
Score: 0.789608717
Page: 91
Text: i. The word log is often used to refer to application logs, where an application outputs text that describes what’s happening. In this book, log is used in the more general sense: an append-only sequence of records. It doesn’t have to be human-readable, it might be binary and intended only for other programs to read. 42,{"name":"San Francisco","attractions":["Golden Gate Bridge"]} 42,{"name":"San Francisco","attractions":["Exploratorium"]} Our db_set function actually has pretty good performance for something that is so simple, because appending to a file is generally very efficient. Many databases internally use a log,i an append-only data file, quite similar to what db_set does. Real databases have more issues to deal with (such as concurrency control, reclaiming disk space so that the log doesn’t grow forever, handling errors, partially written records, and so on) but the basic principle is the same. Logs are incredibly useful, and